# feeding 모델 설명서 — 왜 이렇게 만들었고, 무슨 절차가 일어나는가

이 노트북은 `Feeding_Model.py` 를 그대로 불러와서(**숫자를 다시 쓰지 않고
실제 함수를 실행**해서) 각 설계 결정을 하나씩 증명한다. "왜 이 방법을
골랐는가" 에 대한 답은 전부 이전에 다른 방법을 먼저 시도했다가 실패한
기록이다 — 그 실패들을 같이 남긴다.

## 이 모델이 답하는 질문

feeding(주입) 명령이 나간 뒤:
1. 실제로 액체가 들어왔는가? (**미발생** 여부)
2. 들어왔다면 평소보다 약한가? (**밸브유량**)
3. 밸브가 평소보다 늦게 열리는가? (**지연**)

## 판단 근거를 온도가 아니라 수위로 잡은 이유

처음엔 heat/cool/exchanger 처럼 온도로 접근했다. 그런데 실측하면 feeding 의
Cohen d(효과 크기)가 온도 기준으로는 8개 탱크 중 4개(I1/I2/I3/P2)만
검출 가능했다. **수위**로 바꾸면 8개 전부 d = 2.5~13.9 로, "한 건만 봐도
구분되는" 수준이 된다. 수위는 feeding 의 결과가 직접 찍히는 계측이라
열 지연도, "온도가 낮아서 명령이 나간" 제어 역인과도 안 섞인다.

## 0. 모듈 로드 — 아래 모든 숫자는 이 함수들을 실제로 호출한 결과다

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

# 이 노트북은 parkkt/code_ongoing/ 에 있고 Feeding_Model.py 는 한 단계 위(parkkt/)에 있다.
# (Feeding_Model.py 안의 SAVE_DIR/OUT_DIR/PARAM_PATH 는 __file__ 기준이라 별도 보정이 필요 없다.)
sys.path.insert(0, os.path.dirname(os.getcwd()))
sys.path.insert(0, os.getcwd())

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

import Feeding_Model as FM
from Tank_Temp_Factor_Analysis import TANKS

pd.set_option("display.width", 200)
DEMO_TANK = "I3"

## 1. 수위를 다시 계산하는 이유 — 정수 태그를 안 쓴다

`TK_Level_PV_*` 는 PLC 가 이미 정수로 반올림해서 내놓는 값이다. 원시 아날로그
`Ana_In___LT_*` 에서 직접 계산한 `level_f` 를 쓰면 해상도가 크게 올라간다.

```
level_f = (Ana_In / Ana_Max) × Scale_Max × (Gain/1000) + OffSet
```

하드코딩하지 않고 태그값을 그대로 쓰는 이유: 온도 쪽 `Scale_Max` 가 4월 100 에서
지금 1000 으로 바뀐 전례가 있다 — 파라미터는 언제든 바뀔 수 있다.

In [ ]:
d, src = FM.load(DEMO_TANK)
print(f"[{DEMO_TANK}] 원본 파일: {src}")
print(f"level_f 고유값 {d['level'].nunique()}개  (정수 태그였다면 약 15개 수준)")
print(f"수위 인접 샘플 변화(센서 잡음 추정): 중앙값 {d['level'].diff().abs().median():.3f}")
print("  → 정수 반올림 오차(±0.5)가 실제 잡음(0.04~0.08)보다 훨씬 크다는 뜻")

## 2. 구간 제외 — 00~07시 / 라인 정지 / 알람

```python
d["ok"] = (시간대 제외) & (source != DIAGNOSTIC) & (wagon 이 30분 안에 움직였음)
```

라인 정지는 `Now_Actual_Wagon_Num`(1→42→1 순환 카운터)이 30분간 전혀
안 바뀌면 정지로 판단한다. **2026-09-13 은 하루 종일 라인이 서 있었는데**,
이 필터가 없으면 그 하루가 "정상 운전 데이터"로 섞여 들어가 기준선이 오염된다.

In [ ]:
total = len(d)
excluded_frac = 1 - d["ok"].mean()
print(f"[{DEMO_TANK}] 전체 {total:,}행 중 제외 {excluded_frac*100:.1f}%, 사용 {(1-excluded_frac)*100:.1f}%")

daily = d.groupby(d.index.strftime("%m-%d"))["ok"].mean() * 100
print("\n일별 사용 비율(%):")
print(daily.round(1).to_string())

## 3. 이벤트 단위로 본다 — 5초 격자 통계를 안 쓰는 이유

5초로 리샘플하면 탱크당 수십만 행이 나오지만, 인접 샘플은 거의 같은 값이라
**독립적인 정보량은 그게 아니다.** 실제로 독립적인 관측 단위는 "feeding 동작 한 번"
이다. 그래서 처음부터 이벤트(연속 ON 구간)로 표를 만든다.

**이벤트 최소 길이를 5초로 잡은 이유**: 10초로 하면 I2/I3 의 펄스(중앙값 7.3초)가
통째로 잘려나간다.

In [ ]:
events, src, noise = FM.extract_events2(DEMO_TANK, slope_window=FM.OUTFLOW_PRE_SEC)
print(f"[{DEMO_TANK}] 이벤트 {len(events)}개, 지속시간 중앙값 {events['지속초'].median():.1f}초")
events[["start", "지속초", "유입량", "유출률", "순유량", "밸브유량", "지연초"]].head(8)

## 4. 유출 보정 — 왜 순유량을 그대로 안 쓰나

수위 변화는 `유입 − 유출` 이다. 탱크는 feeding 중에도 계속 소비되고 있어서,
순유량은 **밸브가 낸 유량이 아니다.** feeding 직전 120초의 수위 기울기로
유출률을 재서 뺀다.

```
밸브유량 = 순유량 − 유출률
```

보정으로 얻는 분산 개선은 탱크마다 다르다(P5 21%, 나머지는 1~3%로 작음) —
유출률이 꽤 일정해서 상수 오프셋처럼 작용하기 때문이다. 그래도 넣는 이유는
정확도가 아니라 **견고성**이다: 생산량이 바뀌면 유출률도 바뀌므로, 보정 없는
지표는 밸브가 멀쩡해도 서서히 드리프트해 오경보를 낸다.

In [ ]:
one = events.iloc[len(events) // 2]
print(f"이벤트 예시 ({one['start']}):")
print(f"  유출률(feeding 직전 120초 기울기) = {one['유출률']:+.3f} /분")
print(f"  순유량(끝+60s − 시작−5s, 지속시간으로 나눔) = {one['순유량']:+.3f} /분")
print(f"  밸브유량 = 순유량 − 유출률 = {one['밸브유량']:+.3f} /분   ← 판정에 쓰는 값")

## 5. 기준선 — 중앙값/MAD, 그리고 고정이 아니라 이동

```python
z = (밸브유량 − 직전 30개 중앙값) / (직전 30개 MAD × 1.4826)
```

**왜 평균/표준편차가 아니라 중앙값/MAD 인가.** 고장 이벤트 몇 개가 기준선
자체에 들어가면 평균과 표준편차는 그쪽으로 끌려간다. 중앙값과 MAD(중위절대편차)는
극단값 몇 개에 훨씬 덜 흔들린다.

**왜 고정 기준선이 아니라 이동(직전 30개) 인가.** 4.7일 전체를 고정 기준으로
쓰면, 그 안의 정상적인 공정 변동까지 전부 "이상"으로 누적되어 멀쩡한 탱크가
전부 고장으로 나온다. 릴레이 고장은 서서히가 아니라 **계단형**으로 일어나므로
직전 30개와 비교해도 놓치지 않는다.

**z 를 ±4 로 자르는 이유.** 자르지 않으면 어쩌다 한 번 크게 튄 값 하나가
CUSUM 을 그 즉시 크게 밀어 올려서, 아래 6번의 누적 판정이 무의미해진다.

In [ ]:
v = events["밸브유량"].values.astype(float)
sign = np.sign(np.median(v)) or 1.0
z_raw = FM.rolling_z(v * sign, clip=None) if False else FM.rolling_z(v * sign)
print(f"[{DEMO_TANK}] z 범위: {np.nanmin(z_raw):.2f} ~ {np.nanmax(z_raw):.2f}  (±{FM.Z_CLIP} 로 절단됨)")
print(f"클리핑 안 됐으면 어디까지 갔을지 확인 (window={FM.BASELINE_WINDOW}):")

def mad_noclip(vv, window=FM.BASELINE_WINDOW):
    zz = np.full(len(vv), np.nan)
    for i in range(window, len(vv)):
        med, scale = FM._mad(vv[i-window:i])
        zz[i] = (vv[i] - med) / scale
    return zz

z_unclipped = mad_noclip(v * sign)
print(f"  절단 전 최소값: {np.nanmin(z_unclipped):.2f}  (절단 후: {np.nanmin(z_raw):.2f})")

## 6. CUSUM — 작은 차이가 계속 쌓이는지를 본다

```python
S = max(0, 이전S − z − k)      k = 0.5 (여유분)
```

`max(0, ...)` 는 **"이전 S 와 0 중 큰 것"이 아니라 "계산 결과와 숫자 0 중 큰 것"**
이다. 즉 계산 결과가 음수면 이전에 쌓아둔 값은 **전부 버려지고 0 이 된다.**

`k = 0.5` 는 **둔감대**다. z 가 −0.5 보다 나쁘지 않으면 S 는 오르지 않고
매 이벤트 그만큼씩 깎인다 (z 가 정확히 0 이면 한 스텝에 0.5 씩 감소하지,
즉시 0 이 되는 건 이미 S 가 0.5 이하일 때뿐이다). z 가 계속 −0.5 보다
음수여야만 **누적**된다. 이 방식 덕분에 "한 번 크게 튄 것"과 "작지만 꾸준한 것"을
구분한다 — 후자가 릴레이가 서서히 약해지는 패턴에 가깝다.

**대가가 있다: 0.5σ 미만의 열화는 아무리 오래 지속돼도 영원히 안 잡힌다.**
실제 유량으로 환산하면 평소 밸브유량의 1.6%(I1)~6.1%(P2) 가 사각지대다.
릴레이·솔레노이드 고장은 보통 계단형으로 수십 % 떨어지므로 이 정도는
감수하고, 대신 오경보를 0 으로 유지하는 쪽을 택했다.

**그리고 `Z_CLIP = 4.0` 때문에 한 이벤트가 아무리 심해도 S 는 한 번에
약 3.0~3.6 밖에 못 오른다.** 그래서 완전 정지라도 임계값까지 도달하려면
최소 몇 번의 이벤트가 필요하다 (P1 6회, P5 4회, P2 3회, I3·I1·P3 2회).
"한 번에 크게 떨어지면 잡힌다"가 아니라 **"계속 떨어져야 잡힌다"** 는 뜻이다.

### ⚠️ 이 노트북을 만들며 실제로 고친 버그 — z 중심 편향

z 분포가 평균과 중앙값이 다르면(왼쪽 꼬리가 길면) CUSUM 이 정상 상태에서도
계속 떠오른다. 실측: P5 는 z 중앙값이 −0.01 인데 평균은 −0.66 이었다.
**학습 구간의 z 평균을 `z편향` 으로 저장해서 빼고 넣는다.**

In [ ]:
def cusum_demo(vals, bias):
    zz = FM.rolling_z(vals * sign)
    return FM.cusum_down(zz - bias)

cs_biased = cusum_demo(v, 0.0)
n_tr_demo = int(len(v) * 0.7)
learned_bias = float(np.nanmean(FM.rolling_z(v[:n_tr_demo] * sign)))
cs_fixed = cusum_demo(v, learned_bias)

print(f"[{DEMO_TANK}] z편향 보정 전 CUSUM 평균 {np.nanmean(cs_biased):.1f}, 최대 {np.nanmax(cs_biased):.1f}")
print(f"[{DEMO_TANK}] z편향({learned_bias:+.3f}) 보정 후 CUSUM 평균 {np.nanmean(cs_fixed):.1f}, 최대 {np.nanmax(cs_fixed):.1f}")

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(events["start"], cs_biased, label="보정 전", color="#999999", linewidth=1)
ax.plot(events["start"], cs_fixed, label="보정 후 (실제 사용)", color="#2ca02c", linewidth=1.3)
ax.set_title(f"{DEMO_TANK} — z 중심 보정 전후 CUSUM 비교 (정상 데이터인데도 보정 전엔 계속 떠오른다)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.autofmt_xdate(); fig.tight_layout()

## 7. 임계값 — 목표 오경보 주기(ARL)로 정한다

CUSUM 이 "여기부터 알람" 이라는 선(임계값 h)이 필요하다. 탱크마다 정상 상태에서의
CUSUM 크기 자체가 달라서 하나의 고정값(교과서 기본값 5)은 쓸 수 없다.

### 예전 방식과 그게 틀린 이유

```
(옛날) 임계값 = max(학습 구간 CUSUM 최대값 × 1.3,  5.0)
```

논리는 *"정상 운전만으로 최대 12.11 까지 떠올랐으니, 30% 여유를 두고 15.74 에
선을 긋자"* 였다. 그런데 **"최대값"은 극단값 통계라 표본이 많아질수록 저절로
커진다** — 반에서 제일 키 큰 학생을 재는 것과 같다. 학생이 자란 게 아니라
뽑을 기회가 많아진 것이다. 실측으로 확인된 두 가지 문제:

| 문제 | 실측 |
|---|---|
| **데이터를 모을수록 둔해진다** | P1: 학습 55개 → h=9.4, 148개 → h=16.2. 완전정지 검출이 **3회 → 6회**로 느려짐 |
| **탱크마다 기준이 제각각** | 같은 공식인데 실제 오경보 주기가 P3 1.4일 ~ I1 59일로 **42배** 차이 |

### 지금 방식 — ARL(Average Run Length)

**ARL = "정상 상태에서 알람이 잘못 울릴 때까지 평균 몇 이벤트인가."**

학습 구간의 z 를 복원추출해서 가짜 정상 운전을 4000번 × 20000 이벤트 돌리고,
ARL 이 목표치(`TARGET_ARL = 500`)가 되는 가장 낮은 h 를 고른다. 이건 **분포의
성질**이라 표본 크기에 안 휘둘린다.

> **⚠️ 운영 중 동작은 하나도 안 바뀐다.** 모델은 500 이라는 숫자를 세지도,
> 기다리지도 않는다. 여전히 `S ≥ h` 하나만 본다. 바뀌는 건 **`fit()` 이 h 라는
> 숫자를 고르는 방법**뿐이다. 속도 제한 표지판을 "지난달 제일 빨랐던 차 × 1.3"
> 대신 "사고율이 목표치가 되는 속도"로 정하는 것과 같다 — 운전자가 하는 일은
> 똑같다.

**학습/검증을 반드시 나눈다.** 임계값을 뽑은 바로 그 데이터에서 오경보를
세면 정의상 0 이 나온다(순환논법). `fit()` 은 앞 70%(또는 지정한 시각까지)로
임계값을 정하고, **뒤 30%에서 실제로 오경보가 몇 번 나는지** 센다. 그 숫자만이
이 모델을 믿어도 되는지에 대한 정직한 근거다.

In [ ]:
params, report = FM.fit(train_frac=0.7)
report[["탱크", "전체이벤트", "상태", "학습", "학습z", "검증", "임계값", "검증_오경보", "오경보율", "신뢰도"]]

### 두 방식을 직접 비교 — 임계값이 표본 크기에 휘둘리는가

학습량을 늘려가며 두 방식의 임계값을 나란히 계산한다. **옛 방식은 처음에 하한
5.0 에 붙어 있다가(= 못 믿는 값) 데이터가 쌓이며 기어올라가고, ARL 방식은
처음부터 정착값을 준다.**

In [ ]:
rows = []
for tk in ["P1", "P5", "I3"]:
    e, _, _ = FM.extract_events2(tk, slope_window=FM.OUTFLOW_PRE_SEC)
    v = e["밸브유량"].values.astype(float)
    sign = np.sign(np.median(v)) or 1.0
    for n_tr in [50, 100, 150, 200, 300]:
        if n_tr > len(v) - 20:
            continue
        vt = v[:n_tr] * sign
        bias = float(np.nanmean(FM.rolling_z(vt)))
        z_tr = FM.rolling_z(vt) - bias
        h_old = max(float(np.nanmax(FM.cusum_down(z_tr))) * 1.3, 5.0)   # 옛 방식
        h_arl, _ = FM.threshold_by_arl(z_tr)                            # 지금 방식
        rows.append({"탱크": tk, "학습n": n_tr,
                     "옛방식_h": round(h_old, 1), "하한에걸림": h_old <= 5.01,
                     "ARL방식_h": round(h_arl, 1)})
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))

spread = cmp.groupby("탱크").agg(옛_최소=("옛방식_h", "min"), 옛_최대=("옛방식_h", "max"),
                                  ARL_최소=("ARL방식_h", "min"), ARL_최대=("ARL방식_h", "max"))
spread["옛_변동배수"] = (spread["옛_최대"] / spread["옛_최소"]).round(2)
spread["ARL_변동배수"] = (spread["ARL_최대"] / spread["ARL_최소"]).round(2)
print()
print("=== 학습량에 따른 임계값 변동폭 (1.0 에 가까울수록 안정) ===")
print(spread[["옛_변동배수", "ARL_변동배수"]].to_string())

**"검증_오경보" 열이 이 모델 전체의 핵심 성적표다.** 0 이 아닌 탱크는 아직
신뢰하면 안 된다.

**`신뢰도` 판단 근거도 바뀌었다.** 예전엔 "임계값이 하한 5.0 에 걸렸는가"로
봤는데, ARL 은 z 의 **분포**를 시뮬레이션하므로 못 믿는 경우는 "임계값이 낮다"가
아니라 **분포를 추정할 z 자체가 적다**는 것이다. 그래서 `학습z < 50` 이면
`잠정(z부족)` 으로 표시한다.

⚠️ **학습 이벤트 수 ≠ z 개수**다. 앞 `BASELINE_WINDOW`(30)개는 기준선을 만드는
데 쓰여서 z 가 안 나온다. **I1 은 학습 52개지만 z 는 22개뿐**이고, 이게 I1 의
홀드아웃 오경보 5건(21.7%)의 원인이다 — 방법의 문제가 아니라 데이터가 모자란
것이라, 하루 15.6개씩 나오니 **열흘이면 해결된다.**

## 8. "미발생" 게이트 — CUSUM 과 별개의 즉시 판정

```python
발생 = 유입량 ≥ 그 탱크 유입량 중앙값 × 0.25
```

CUSUM 은 여러 이벤트가 쌓여야 반응하는데, "아예 안 들어왔다"는 즉시 알아야
한다. 0.25 라는 기준은 센서 잡음과 정상 변동 사이의 빈 공간에 그은 선이다 —
잡음(수위 변화 0.04~0.08)의 수십 배 위이면서, 정상 변동폭(변동계수 0.08~0.30)
보다는 한참 아래다.

In [ ]:
p = params[DEMO_TANK]
thr_occurred = p["유입량중앙"] * FM.OCCURRED_FRAC
print(f"[{DEMO_TANK}] 유입량 중앙값 {p['유입량중앙']:.2f}  ×  {FM.OCCURRED_FRAC}  =  미발생 기준선 {thr_occurred:.2f}")
print(f"  수위 센서 잡음: {p['수위잡음']:.3f}  (기준선의 {thr_occurred/p['수위잡음']:.0f}분의 1 수준)")

## 9. 표본 부족 탱크(P4·I2) — 이력 없이 되는 절대판정

CUSUM 은 이동 기준선(직전 30개 이벤트)이 있어야 도는데, P4(6건)·I2(38건)는
그 전이다. 예전엔 이 두 탱크가 `check()` 목록에 아예 안 나타났다 — 릴레이가
완전히 죽어도 알아챌 방법이 없었다는 뜻이다.

**"완전 미발생"만큼은 이력 없이 이벤트 하나로 바로 판정할 수 있다.** 이벤트
구간의 수위 샘플 개수를 알면, 순수 잡음만으로 우연히 나올 수 있는 흔들림
크기를 확산 근사로 계산할 수 있다:

```python
잡음에_의한_예상흔들림 = 수위잡음 × √(그 구간 샘플 수)
잡음배수 = |실제 유입량| / 잡음에_의한_예상흔들림
발생_절대 = 잡음배수 >= 3.0
```

**한계도 분명하다.** 이 방법으로는 "완전히 안 들어왔는지"만 잡는다.
"평소보다 30% 약해졌는지" 같은 정도의 문제는 여전히 못 잡는다 — 그건
"이 탱크 밸브의 평소 유량이 얼마인지"를 알아야 하는데, 그건 여러 이벤트를
봐야만 알 수 있다.

In [ ]:
for tk in ["P4", "I2"]:
    e, src, noise = FM.extract_events2(tk, slope_window=FM.OUTFLOW_PRE_SEC)
    print(f"[{tk}] 이벤트 {len(e)}개 (CUSUM 에 필요한 {FM.MIN_EVENTS_TO_FIT}개에는 못 미침)")
    print(e[["start", "지속초", "유입량", "잡음배수", "발생_절대"]].to_string(index=False))
    print()

## 10. 민감도 검증 — 실제 고장이 없으니 가짜로 주입해서 잰다

마지막 20개 이벤트의 밸브유량에 `× (1 − 심각도)` 를 곱해서 반응을 약화시키고,
몇 번째 이벤트에서 CUSUM 이 임계값을 넘는지 본다. **100% = 완전 정지,
25% = 부분 열화(필터 막힘 등)** 시뮬레이션이다.

In [ ]:
sens = FM.sensitivity(params)
sens

## 11. CL(닫힘) 딜레이 — 별도 조기 경보 (CUSUM 이 아니라 단순 카운트)

밸브가 닫으라는 명령을 받고도 아직 열림 상태에 붙어 있는 시간이 **딜레이**다.
0/1/2 값으로 찍히고, 실제 알람은 이게 **3초를 넘을 때** 울리는데 지금까지
한 번도 넘은 적이 없다. 2초는 그 바로 직전 단계다.

**왜 CUSUM 이 아니라 단순 누적 카운트인가.** 딜레이는 밸브유량보다 변동계수가
2~6배 커서(잡음이 훨씬 심해서) CUSUM 을 그대로 붙이면 P2·I1 같은 탱크에서
오경보가 났다(실측). 그리고 **"한두 번은 정상 변동일 수 있다"** — P4(1건),
I2(2건)는 baseline 잡음 수준이었다. 그래서 최근 7일 안에 **3회 이상 누적**
됐을 때만 "확인요망"으로 표시한다. P3/I1/I3 는 5일간 0건이라, 이 셋 중
하나가 2초를 찍기 시작하는 것 자체가 이미 이례적이다.

In [ ]:
delay_report = FM.close_delay_report()
delay_report

## 12. 최종 판정 — check()

저장된 파라미터로 현재 데이터를 판정한다. 판정은 4단계다.

| 판정 | 조건 |
|---|---|
| **미발생** | 유입량이 게이트(0.25×중앙값) 미만 |
| **확정** | CUSUM ≥ 임계값 |
| **주의** | CUSUM ≥ 임계값×0.6, 또는 이번 이벤트 z ≤ −3 |
| **정상** | 그 외 |

**표본이 부족한 탱크는 이 4단계 대신 "정상/미발생" 2단계(절대판정만)로
나온다.** `모드` 열이 `CUSUM` 인지 `절대판정만` 인지로 구분된다 — 후자는
"완전 미발생"만 보장하지, "확정/주의" 단계(서서히 약해짐)는 못 낸다.

In [ ]:
result = FM.check()

## 13. 운영 — 앞으로 이렇게 쓴다

```bash
# 새 데이터 받기
python Tank_Temp_Factor_Analysis.py --start "2026-09-09 19:00:00"

# 판정만 (평소에 이것만 돌리면 된다)
python Feeding_Model.py check

# 파라미터 재학습 (기준선 자체를 다시 잡음 — 매번 할 필요는 없다)
python Feeding_Model.py fit
```

**`fit` 을 가끔 다시 돌려서 볼 것 — 홀드아웃 오경보율이 계속 0% 로 유지되는가.**
이게 이 모델을 계속 믿어도 되는지에 대한 유일한 지표다. 오경보가 나기
시작하면 공정 자체가 바뀐 것(레시피 변경 등)일 수 있다.

## 14. 한계 — 이 모델이 못 하는 것

- **P4(6건)·I2(38건) 는 여전히 CUSUM(서서히 약해짐 감지)이 안 된다.** 위
  9번에서 넣은 절대판정 덕분에 "완전 미발생"만큼은 이제 감시되지만, 그 이상
  (평소보다 30% 약해졌다 같은 정도)은 이동 기준선(30개)이 쌓여야 한다.
- **야간(00~07시)·재가동 직후는 학습 대상이 아니다.** 그 구간에 적용하면
  오경보가 날 수 있다.
- **원인은 구분 못 한다.** "명령 대비 결과가 안 나왔다" 까지만 말한다 —
  릴레이인지 솔레노이드인지 배관 막힘인지는 현장 확인이 필요하다.
- **레시피가 바뀌면 이동 기준선이 따라잡을 때까지(약 30 이벤트) 오경보 가능.**